# 第 27 天：ML因子2

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：ML因子2
> 必做：LightGBM
> 选做：XGBoost
> 目标产出：特征重要性

> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。

## 0. 今天你要真正学会什么？

1. 理解树模型和线性模型在因子预测里的差异。
2. 训练一个可复现的基线模型，并做样本外评估。
3. 用排列重要性观察特征贡献，建立特征筛选思路。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

模型像一个聪明但不一定诚实的助手。你不能只问它预测是多少，还要问它为什么这么预测，以及这个理由在样本外是否站得住。

## 5. 今日核心实验


### 实验 1：准备训练集和测试集

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
feature_names = ["value", "quality", "growth", "momentum_20", "momentum_60", "low_vol", "liquidity", "reversal_5", "price_volume"]
panel = build_panel({name: factor_library[name] for name in feature_names}, future_5d)
panel["value_x_quality"] = panel["value"] * panel["quality"]
panel["momentum_x_liquidity"] = panel["momentum_20"] * panel["liquidity"]
panel["short_long_mom_gap"] = panel["momentum_20"] - panel["momentum_60"]

train, test, split_date = time_split_panel(panel, 0.7)
features = [c for c in panel.columns if c != "label"]
X_train, y_train = train[features].to_numpy(), train["label"].to_numpy()
X_test, y_test = test[features].to_numpy(), test["label"].to_numpy()

print("切分日期：", split_date.date())
print("训练/测试：", X_train.shape, X_test.shape)


### 实验 2：线性 Ridge 基线：先有一个稳得住的参照物

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
coef = fit_ridge(X_train, y_train, lam=10.0)
pred_train = predict_ridge(X_train, coef)
pred_test = predict_ridge(X_test, coef)

def corr_score(y_true, y_pred):
    return pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")

print({
    "train_rank_corr": round(float(corr_score(y_train, pred_train)), 4),
    "test_rank_corr": round(float(corr_score(y_test, pred_test)), 4),
})


### 实验 3：可选树模型接口：有 LightGBM/XGBoost 就用，没有也不阻塞学习

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
model_note = "本机未安装 LightGBM/XGBoost，课程使用 Ridge 基线和排列重要性完成同样的研究流程。"
try:
    import lightgbm as lgb
    model_note = "检测到 LightGBM，可在真实环境中替换 Ridge 基线。"
except Exception:
    try:
        import xgboost as xgb
        model_note = "检测到 XGBoost，可在真实环境中替换 Ridge 基线。"
    except Exception:
        pass

print(model_note)


### 实验 4：排列重要性：不要只听模型自己吹

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
baseline = corr_score(y_test, pred_test)
rng = np.random.default_rng(202630)
importance_rows = []

for idx, name in enumerate(features):
    X_perm = X_test.copy()
    rng.shuffle(X_perm[:, idx])
    perm_pred = predict_ridge(X_perm, coef)
    score = corr_score(y_test, perm_pred)
    importance_rows.append({
        "feature": name,
        "baseline": baseline,
        "permuted_score": score,
        "importance": baseline - score,
    })

importance = pd.DataFrame(importance_rows).set_index("feature").sort_values("importance", ascending=False)
print(importance.round(5))


### 实验 5：预测值变成因子：模型输出也要走 IC 检验

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
test_pred = pd.Series(pred_test, index=test.index, name="ml_score")
ml_score = test_pred.unstack("asset")
test_label = future_5d.reindex(ml_score.index)

ml_report = factor_report(ml_score, test_label, "ml_score")
print(ml_report.drop("factor").astype(float).round(4))


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：ML因子2
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：调参调到样本内很好，样本外崩掉。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：只看模型分数，不看预测值是否能排序选股。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：把特征重要性当因果解释。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：忽略模型每期重新训练的成本和稳定性。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 28 天会把 ML 预测变成真正的选股组合。

## 13. 一句话收尾

ML因子2 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒（更新版）

今天如果只记住一件事，记住这个：**排列重要性告诉你"哪个特征有用"，SHAP告诉你"特征怎么起作用"，IC衰减告诉你"模型有没有过拟合"。** 三者缺一不可。

---

## 15. 进阶：模型对比 —— 不止跑一个模型

### 15.1 多模型对比框架


In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    """统一评估接口"""
    model.fit(X_tr, y_tr)
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    
    tr_ic = pd.Series(pred_tr).corr(pd.Series(y_tr), method="spearman")
    te_ic = pd.Series(pred_te).corr(pd.Series(y_te), method="spearman")
    
    return {
        "model": name,
        "train_ic": round(tr_ic, 4),
        "test_ic": round(te_ic, 4),
        "ic_gap": round(tr_ic - te_ic, 4),
    }

models = {
    "Ridge (alpha=1)": Ridge(alpha=1.0),
    "Ridge (alpha=10)": Ridge(alpha=10.0),
    "Lasso (alpha=0.001)": Lasso(alpha=0.001, max_iter=5000),
    "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
}

results = []
for name, model in models.items():
    try:
        result = evaluate_model(model, X_train, y_train, X_test, y_test, name)
        results.append(result)
    except Exception as e:
        results.append({"model": name, "error": str(e)})

comparison = pd.DataFrame(results)
print("模型对比结果：")
print(comparison)


### 15.2 树模型 vs 线性模型：金融场景的选择

| 维度 | 线性模型 (Ridge/Lasso) | 树模型 (LGB/XGB) |
|------|------------------------|-------------------|
| 可解释性 | ★★★★★ 系数直接可读 | ★★☆ 需要 SHAP 辅助 |
| 非线性捕捉 | ★☆☆ 需要手工交互特征 | ★★★★ 自动学习交互 |
| 过拟合风险 | ★★★ 中等 | ★★★★★ 非常高 |
| 训练速度 | ★★★★★ 极快 | ★★★ 中等 |
| 调参难度 | ★★ 简单 | ★★★★ 参数多 |
| 样本外稳定性 | ★★★★ 较好 | ★★☆ 易退化 |

**经验法则**：先用 Ridge 建立基线 → 确认特征工程有效 → 再尝试树模型 → 用 CV 严格评估。

---

## 16. 进阶：排列重要性 vs SHAP —— 特征解释的两种武器

### 16.1 SHAP 值（比排列重要性更精细）


In [ ]:
# 如果安装了 shap 库
try:
    import shap
    
    # 对 Ridge 模型计算 SHAP
    explainer = shap.LinearExplainer(
        (coef[1:], coef[0]),  # (系数, 截距)
        X_train[:500]  # 用训练集子集做背景
    )
    shap_values = explainer.shap_values(X_test[:500])
    
    # 计算平均绝对SHAP值
    shap_importance = pd.DataFrame({
        "feature": features,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)
    print("SHAP 特征重要性排名：")
    print(shap_importance.head(10))
    
except ImportError:
    print("未安装 shap 库。安装命令：pip install shap")
    print("排列重要性（实验4）已经是很好的替代方案。")


### 16.2 对比：排列重要性 vs SHAP

| 特性 | 排列重要性 | SHAP |
|------|-----------|------|
| 模型无关 | ✓ 任何模型 | 需要专门Explainer |
| 计算速度 | 慢（需多次重排预测） | 快（一次性计算） |
| 特征交互 | 无法分解 | 可分解交互效应 |
| 方向性 | 只有重要性大小 | 有正负方向 |

---

## 17. 进阶：滚动窗口评估 —— 更接近实盘


In [ ]:
def rolling_window_backtest(X, y, dates, model_func, train_window=252, retrain_freq=21):
    """
    滚动窗口回测：模拟实盘中定期重新训练的场景。
    train_window=252: 训练窗口（约1年）
    retrain_freq=21: 重新训练频率（约1个月）
    """
    unique_dates = sorted(dates.unique())
    results = []
    
    for start_idx in range(0, len(unique_dates) - train_window - 21, retrain_freq):
        train_end = unique_dates[start_idx + train_window - 1]
        test_start = unique_dates[start_idx + train_window]
        test_end = unique_dates[min(start_idx + train_window + 20, len(unique_dates) - 1)]
        
        train_mask = (dates >= unique_dates[start_idx]) & (dates <= train_end)
        test_mask = (dates >= test_start) & (dates <= test_end)
        
        X_tr = X[train_mask.values]
        y_tr = y[train_mask.values]
        X_te = X[test_mask.values]
        y_te = y[test_mask.values]
        
        if len(X_tr) < 100 or len(X_te) < 5:
            continue
        
        coef = model_func(X_tr, y_tr)
        pred = predict_ridge(X_te, coef)
        ic = pd.Series(pred).corr(pd.Series(y_te), method="spearman")
        
        results.append({
            "train_end": train_end, "ic": ic,
            "n_train": len(X_tr), "n_test": len(X_te),
        })
    
    return pd.DataFrame(results)

# 使用示例（需匹配日期索引）
# roll_results = rolling_window_backtest(X_train, y_train, train["date"], fit_ridge)
# print(f"滚动窗口平均IC: {roll_results['ic'].mean():.4f}")
# print(f"IC为正的窗口占比: {(roll_results['ic'] > 0).mean():.1%}")


---

## 18. 进阶：过拟合检测的量化指标


In [ ]:
def overfit_diagnosis(train_pred, test_pred, y_train, y_test, n_features):
    """量化过拟合程度"""
    
    tr_ic = pd.Series(train_pred).corr(pd.Series(y_train), method="spearman")
    te_ic = pd.Series(test_pred).corr(pd.Series(y_test), method="spearman")
    ic_decay = (tr_ic - te_ic) / max(abs(tr_ic), 1e-8)
    
    var_ratio = np.var(test_pred) / max(np.var(train_pred), 1e-8)
    param_ratio = n_features / len(y_train)
    
    report = {
        "train_ic": round(tr_ic, 4),
        "test_ic": round(te_ic, 4),
        "ic_decay_rate": round(ic_decay, 3),
        "pred_var_ratio": round(var_ratio, 3),
        "param_ratio": round(param_ratio, 4),
    }
    
    warnings = []
    if abs(ic_decay) > 0.3:
        warnings.append(f"⚠️ IC衰减率 {ic_decay:.1%}，严重过拟合")
    if var_ratio < 0.3:
        warnings.append("⚠️ 测试集预测方差远小于训练集")
    if param_ratio > 0.02:
        warnings.append(f"⚠️ 参数/样本比 {param_ratio:.3f} 偏高")
    
    report["warnings"] = warnings if warnings else ["✓ 未检测到明显过拟合"]
    return report

# diag = overfit_diagnosis(pred_train, pred_test, y_train, y_test, len(features))
# for k, v in diag.items():
#     print(f"{k}: {v}")


---

本课程内容仅用于量化研究学习，不构成投资建议。
